<a href="https://colab.research.google.com/github/busycaesar/Attention_Attention_Everywhere/blob/Master/QKV-new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers

## Get Model

In [2]:
from transformers import GPT2Tokenizer, GPT2Model

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2Model.from_pretrained('gpt2', output_attentions=True)
model.eval()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Model(
  (wte): Embedding(50257, 768)
  (wpe): Embedding(1024, 768)
  (drop): Dropout(p=0.1, inplace=False)
  (h): ModuleList(
    (0-11): 12 x GPT2Block(
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): GPT2Attention(
        (c_attn): Conv1D(nf=2304, nx=768)
        (c_proj): Conv1D(nf=768, nx=768)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): GPT2MLP(
        (c_fc): Conv1D(nf=3072, nx=768)
        (c_proj): Conv1D(nf=768, nx=3072)
        (act): NewGELUActivation()
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
)

## Series of tokens

In [3]:
sentence = 'The dog did not cross the road because it was'

inputs = tokenizer(sentence, return_tensors='pt')

print(inputs)

{'input_ids': tensor([[ 464, 3290,  750,  407, 3272,  262, 2975,  780,  340,  373]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


## List of tokens with probability

In [18]:
import torch

with torch.no_grad():
    h = model(**inputs).last_hidden_state[0, -1]

logits = h @ model.wte.weight.T
probs  = torch.softmax(logits, dim=-1)

top = torch.topk(probs, 50)
for p, i in zip(top.values, top.indices):
    print(f'{repr(tokenizer.decode([i])):>14}  {p.item()*100:6.2f}%')

     ' afraid'    7.08%
        ' not'    6.69%
        ' too'    6.58%
     ' scared'    5.02%
         ' in'    2.55%
          ' a'    2.08%
 ' frightened'    1.80%
    ' running'    1.65%
         ' so'    1.63%
     ' trying'    1.51%
         ' on'    1.43%
          ' "'    1.41%
      ' being'    1.23%
    ' fearful'    1.20%
    ' worried'    1.12%
    ' barking'    0.87%
  ' concerned'    0.83%
       ' very'    0.70%
        ' out'    0.69%
      ' still'    0.68%
    ' chasing'    0.65%
     ' unable'    0.64%
    ' already'    0.60%
      ' going'    0.60%
      ' under'    0.59%
      ' stuck'    0.59%
    ' injured'    0.55%
       ' just'    0.50%
     ' caught'    0.47%
  ' terrified'    0.45%
     ' moving'    0.44%
         ' at'    0.42%
       ' safe'    0.42%
     ' hungry'    0.41%
    ' getting'    0.40%
    ' waiting'    0.39%
 ' distracted'    0.39%
       ' over'    0.37%
      ' doing'    0.36%
         ' un'    0.36%
   ' carrying'    0.36%
      ' angry'  

In [5]:
tokens = [tokenizer.decode([tid]) for tid in inputs['input_ids'][0]]

it_idx   = tokens.index(' it')

print(it_idx)

8


In [6]:
print('GPT-2 loaded.')
print(f'Layers : {model.config.n_layer}')
print(f'Heads  : {model.config.n_head}')
print(f'd_model: {model.config.n_embd}')
print(f'd_k    : {model.config.n_embd // model.config.n_head}  (d_model / n_heads)')

GPT-2 loaded.
Layers : 12
Heads  : 12
d_model: 768
d_k    : 64  (d_model / n_heads)


In [12]:
import torch

qkv_store = {}

def make_hook(layer_idx):
    def hook(module, input, output):
        with torch.no_grad():
            x   = input[0]
            qkv = module.c_attn(x)
            d   = model.config.n_embd
            Q, K, V = qkv.split(d, dim=2)
            qkv_store[layer_idx] = (
                Q.squeeze(0).detach(),
                K.squeeze(0).detach(),
                V.squeeze(0).detach()
            )
    return hook

with torch.no_grad():
    outputs = model(**inputs)

def fmt(v, n=4):
    return '[' + ', '.join(f'{x:+.3f}' for x in v[:n].tolist()) + ', ...]'

def fmt_ends(v, n=3):
    h = ', '.join(f'{x:+.3f}' for x in v[:n].tolist())
    t = ', '.join(f'{x:+.3f}' for x in v[-n:].tolist())
    return f'[{h}, ... , {t}]   (len {len(v)})'

with torch.no_grad():
    hs = model(**inputs, output_hidden_states=True).hidden_states

n_layers = model.config.n_layer
n_heads  = model.config.n_head
d_head   = model.config.n_embd // n_heads
prior    = list(range(it_idx + 1))
names    = [tokens[i].strip() for i in prior]
blocks   = model.transformer.h if hasattr(model, 'transformer') else model.h

print('='*100)
print('Starting vector for "it"  (token embedding + positional encoding)')
print('='*100)
print(f'  x_it = {fmt(hs[0][0, it_idx], 10)}   (768 numbers total)')
print('\n  note: each layer also has an MLP sublayer that adds to x_it.')
print('        the tables below trace the attention half only.\n')

for L in range(n_layers):
    print('\n' + '='*100)
    print(f'LAYER {L}')
    print('='*100)

    hdr = f'{"Head":<6}' + ''.join(f'{n:>10}' for n in names)
    print(hdr); print('-'*len(hdr))
    for H in range(n_heads):
        w = outputs.attentions[L][0][H][it_idx][:len(prior)]
        print(f'{H:<6}' + ''.join(f'{v:>10.3f}' for v in w.tolist()))

    _, _, V = qkv_store[L]
    head_outs = []
    for H in range(n_heads):
        w  = outputs.attentions[L][0][H][it_idx]
        Vh = V[:, H*d_head:(H+1)*d_head]
        head_outs.append((w.unsqueeze(1) * Vh).sum(0))

    # ---- show the formula with real numbers for head 0 ----
    w0  = outputs.attentions[L][0][0][it_idx]
    V0  = V[:, 0:d_head]
    print(f'\n  head output = sum of (attention weight x value vector)')
    print(f'  head 0, written out:\n')
    for i in prior:
        print(f'      {w0[i]:+.3f}  x  v_{names[i]:<8} {fmt(V0[i], 3)}')
    print(f'      {"-"*54}')
    print(f'      sum     = {fmt(head_outs[0], 3)}   ({d_head} numbers)')

    z = torch.cat(head_outs)
    print(f'\n  all {n_heads} heads flattened:')
    print(f'      z = {fmt_ends(z)}')

    Wo = blocks[L].attn.c_proj.weight
    bo = blocks[L].attn.c_proj.bias

    print(f'\n  W^O  — one per layer, shared by all {n_heads} heads   (top-left 4x4 of 768x768)')
    for r in range(4):
        print('      ' + ''.join(f'{Wo[r, c]:>9.3f}' for c in range(4)))

    attn_out = z @ Wo + bo
    print(f'\n  z @ W^O  — each output slot is z dotted with one column of W^O')
    terms = ' + '.join(f'({z[k]:+.3f})({Wo[k,0]:+.3f})' for k in range(3))
    print(f'      slot 0 = {terms} + ... + b')
    print(f'             = {attn_out[0]:+.3f}')
    print(f'      attention output = {fmt(attn_out)}')

    x_old = hs[L][0, it_idx]
    print(f'\n  new x_it = old x_it + attention output')
    print(f'      old x_it     {fmt(x_old)}')
    print(f'    + attn output  {fmt(attn_out)}')
    print(f'    ' + '-'*46)
    print(f'    = new x_it     {fmt(x_old + attn_out)}')

Starting vector for "it"  (token embedding + positional encoding)
  x_it = [+0.025, -0.048, +0.144, +0.046, -0.066, -0.020, -0.252, +0.053, +0.045, +0.048, ...]   (768 numbers total)

  note: each layer also has an MLP sublayer that adds to x_it.
        the tables below trace the attention half only.


LAYER 0
Head         The       dog       did       not     cross       the      road   because        it
------------------------------------------------------------------------------------------------
0          0.336     0.158     0.080     0.094     0.092     0.043     0.048     0.087     0.061
1          0.008     0.001     0.001     0.006     0.000     0.018     0.000     0.002     0.964
2          0.286     0.074     0.169     0.069     0.026     0.052     0.067     0.201     0.054
3          0.007     0.001     0.001     0.002     0.003     0.013     0.010     0.031     0.932
4          0.089     0.021     0.064     0.054     0.072     0.047     0.157     0.267     0.230
5       